In [1]:
from transformers import AutoTokenizer, AutoConfig, EncoderDecoderModel, PretrainedConfig
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import sqrt

In [2]:
model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = BertModel.from_pretrained(model_ckpt)

In [3]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): BertLayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (intermediate): BertIntermediate(
          (den

In [4]:
text = "time flies like an arrow"
show(model, "bert", tokenizer, text, display_mode = "light", layer=0, head=8)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Encoder

This notebook builds a transformer **encoder** block from scratch with plain `torch.nn` modules, mirroring `bert-base-uncased`'s architecture (loaded above only to reuse its `config`, tokenizer, and hidden sizes, and to visualize attention with `bertviz`).

The Encoder self-attention is **bidirectional**: every token can attend to every other token, so no causal mask is needed anywhere below.

The build proceeds bottom-up:
1. Manually compute (unmasked) scaled dot-product attention on raw embeddings.
2. Wrap that logic into a reusable `scaled_dot_product_attention` function.
3. Compose it into `AttentionHead` → `MultiHeadAttention`.
4. Add learned token + positional `Embeddings`.
5. Add a `FeedForward` block and combine everything into a pre-norm `TransformerEncoderLayer`.
6. Stack everything into a full `TransformerEncoder`.

In [5]:
inputs = tokenizer(text, return_tensors = "pt", add_special_tokens = False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

In [6]:
config = AutoConfig.from_pretrained(model_ckpt)
token_emb = nn.Embedding(config.vocab_size, config.hidden_size)
token_emb

Embedding(30522, 768)

In [7]:
inputs_embeds = token_emb(inputs.input_ids)
inputs_embeds.size()

torch.Size([1, 5, 768])

### Manual attention

Compute self-attention step by step on the raw token embeddings, without a function yet:

- Compute raw attention **scores** as scaled dot products between queries and keys.
- Turn the scores into **weights** with softmax (no masking — each row sums to 1 over the *whole* sequence, since every position may attend to every other position).
- Take the weighted sum of values to get the attention output.

In [8]:
query = key = value = inputs_embeds
dim_k = key.size(-1)

scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
scores.size()

torch.Size([1, 5, 5])

In [9]:
weights = F.softmax(scores, dim=-1)
weights.sum(dim=-1)

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)

In [10]:
attention_outputs = torch.bmm(weights, value)
attention_outputs.shape

torch.Size([1, 5, 768])

### Reusable attention function

The manual steps above are packaged into `scaled_dot_product_attention`, so the rest of the notebook can call it instead of repeating the score/softmax logic every time.

In [11]:
def scaled_dot_product_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
    """Compute scaled dot-product attention.

    Unlike the decoder's version, this takes no mask: encoder self-attention
    is bidirectional, so every position may attend to every other position.

    Args:
        query: Query tensor of shape (batch_size, seq_len, head_dim).
        key: Key tensor of shape (batch_size, seq_len, head_dim).
        value: Value tensor of shape (batch_size, seq_len, head_dim).

    Returns:
        Attention output tensor of shape (batch_size, seq_len, head_dim).
    """
    dim_k = query.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / sqrt(dim_k)
    weights = F.softmax(scores, dim=-1)
    return torch.bmm(weights, value)

### Attention head

`AttentionHead` adds the missing piece from the manual demo above: **learned** linear projections (`q`, `k`, `v`) applied to the hidden state before attention, instead of using the raw embeddings directly as query/key/value.

In [12]:
class AttentionHead(nn.Module):
    """A single scaled dot-product attention head."""

    def __init__(self, embed_dim: int, head_dim: int) -> None:
        """Initialize the query, key, and value projections.

        Args:
            embed_dim: Dimensionality of the input embeddings.
            head_dim: Dimensionality of this head's projected space.
        """
        super().__init__()
        self.q = nn.Linear(embed_dim, head_dim)
        self.k = nn.Linear(embed_dim, head_dim)
        self.v = nn.Linear(embed_dim, head_dim)

    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        """Project the input and apply scaled dot-product attention.

        Args:
            hidden_state: Input tensor of shape (batch_size, seq_len, embed_dim).

        Returns:
            Attention output tensor of shape (batch_size, seq_len, head_dim).
        """
        return scaled_dot_product_attention(
            self.q(hidden_state), self.k(hidden_state), self.v(hidden_state)
        )

### Multi-head attention

Instead of a single attention head, `MultiHeadAttention` runs `num_attention_heads` heads in parallel (each with `head_dim = embed_dim / num_heads`), concatenates their outputs, and projects the result back to `embed_dim` with `output_linear`. This lets the model attend to different representation subspaces at once.

In [13]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention block combining several AttentionHead outputs."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the attention heads and output projection.

        Args:
            config: Model config exposing hidden_size and num_attention_heads.
        """
        super().__init__()
        embed_dim = config.hidden_size
        num_heads = config.num_attention_heads
        head_dim = embed_dim // num_heads
        self.heads = nn.ModuleList(
            [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
        )
        self.output_linear = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, hidden_state: torch.Tensor) -> torch.Tensor:
        """Run all attention heads in parallel and merge their outputs.

        Args:
            hidden_state: Input tensor of shape (batch_size, seq_len, embed_dim).

        Returns:
            Output tensor of shape (batch_size, seq_len, embed_dim).
        """
        x = torch.cat([h(hidden_state) for h in self.heads], dim = -1)
        return self.output_linear(x)

### Embeddings

`Embeddings` turns input token ids into vectors by summing a learned **token embedding** with a learned **positional embedding** (based on each token's index in the sequence), then applies `LayerNorm` and dropout. This gives the model both content and order information, since attention itself is permutation-invariant.

This class originally referenced `config.decoder.hidden_size` / `config.decoder.max_position_embeddings` — leftover from the decoder notebook this one was adapted from. Since `config` here is a flat `BertConfig` (no `.decoder` nesting), that raised an `AttributeError`; it's been fixed to use `config.hidden_size` / `config.max_position_embeddings` directly.

In [14]:
class Embeddings(nn.Module):
    """Token + learned positional embeddings with layer norm and dropout."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the token and position embedding tables.

        Args:
            config: Model config exposing vocab_size, hidden_size, and
                max_position_embeddings.
        """
        super().__init__()
        self.token_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        self.position_embeddings = nn.Embedding(config.max_position_embeddings, config.hidden_size)
        self.layer_norm = nn.LayerNorm(config.hidden_size, eps=1e-12)
        self.dropout = nn.Dropout()

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Embed input token ids and add positional information.

        Args:
            input_ids: Tensor of token ids with shape (batch_size, seq_len).

        Returns:
            Embedding tensor of shape (batch_size, seq_len, hidden_size).
        """
        # Create position ids for input sequence
        seq_lenght = input_ids.size(1)
        position_ids = torch.arange(seq_lenght, dtype=torch.long,).unsqueeze(0)

        # Create token and position embeddings
        token_embeddings = self.token_embeddings(input_ids)
        position_embeddings = self.position_embeddings(position_ids)

        # Combinate token and position embeddings
        embeddings = token_embeddings + position_embeddings
        embeddings = self.layer_norm(embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings

### Quick check: multi-head attention

Sanity-check `MultiHeadAttention` on its own by running it directly on `inputs_embeds` (before there's a full encoder to run it inside).

In [15]:
multihead_attention = MultiHeadAttention(config)
attn_outputs = multihead_attention(inputs_embeds)
attn_outputs.size()

torch.Size([1, 5, 768])

### Feed-forward block

`FeedForward` is the position-wise sublayer applied independently to each token: expand `hidden_size` → `intermediate_size` with a GELU nonlinearity, then project back down to `hidden_size`, with dropout for regularization.

In [16]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network used inside a transformer layer."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the two linear layers, activation, and dropout.

        Args:
            config: Model config exposing hidden_size, intermediate_size,
                and hidden_dropout_prob.
        """
        super().__init__()
        self.linear_1 = nn.Linear(config.hidden_size, config.intermediate_size)
        self.linear_2 = nn.Linear(config.intermediate_size, config.hidden_size)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply the two-layer feed-forward transform.

        Args:
            x: Input tensor of shape (batch_size, seq_len, hidden_size).

        Returns:
            Output tensor of the same shape as the input.
        """
        x = self.linear_1(x)
        x = self.gelu(x)
        x = self.linear_2(x)
        x = self.dropout(x)
        return x

### Encoder layer

`TransformerEncoderLayer` combines `MultiHeadAttention` and `FeedForward` into a **pre-norm** transformer block: `LayerNorm` is applied *before* each sublayer, and the sublayer's output is added back via a residual connection (`x = x + sublayer(norm(x))`). No mask is passed to `self.attention`, since encoder self-attention is bidirectional by design.

In [17]:
class TransformerEncoderLayer(nn.Module):
    """A single pre-norm transformer encoder layer (self-attention + feed-forward)."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the layer norms, attention block, and feed-forward block.

        Args:
            config: Model config exposing hidden_size and the fields
                required by MultiHeadAttention and FeedForward.
        """
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(config.hidden_size)
        self.layer_norm_2 = nn.LayerNorm(config.hidden_size)
        self.attention = MultiHeadAttention(config)
        self.feed_forward = FeedForward(config)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply self-attention and feed-forward sublayers with residual connections.

        Args:
            x: Input tensor of shape (batch_size, seq_len, hidden_size).

        Returns:
            Output tensor of the same shape as the input.
        """
        hidden_state = self.layer_norm_1(x)
        x = x + self.attention(hidden_state)
        x = x + self.feed_forward(self.layer_norm_2(x))
        return x

### Full encoder stack

`TransformerEncoder` puts everything together: it embeds the input ids, then runs them through `config.num_hidden_layers` stacked `TransformerEncoderLayer`s — no mask needed at any layer, unlike the decoder stack, where a causal mask has to be threaded through.

In [18]:
class TransformerEncoder(nn.Module):
    """Stack of transformer encoder layers on top of token + position embeddings."""

    def __init__(self, config: PretrainedConfig) -> None:
        """Build the embedding layer and the stack of encoder layers.

        Args:
            config: Model config exposing num_hidden_layers plus the fields
                required by Embeddings and TransformerEncoderLayer.
        """
        super().__init__()
        self.embeddings = Embeddings(config)
        self.layers = nn.ModuleList(
            [TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Embed the input and pass it through the stacked encoder layers.

        Args:
            x: Input token ids of shape (batch_size, seq_len).

        Returns:
            Output tensor of shape (batch_size, seq_len, hidden_size).
        """
        x = self.embeddings(x)
        for layer in self.layers:
            x = layer(x)
        return x

### Sanity check

Instantiate the encoder from `config` and run a forward pass on the tokenized input. The output shape `(batch_size, seq_len, hidden_size)` confirms the whole stack — embeddings, bidirectional multi-head attention, and feed-forward layers — runs end to end.

In [19]:
encoder = TransformerEncoder(config)
encoder(inputs.input_ids).size()

torch.Size([1, 5, 768])